# Retrieval Augmented Generation with Langchain
- data -> document -> chunks -> embeddings -> 

In [ ]:
# !pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

## 1. Creating Document

In [2]:
from langchain_core.documents import Document

In [3]:
sample_doc = Document(
    page_content = "Hello World !",
    metadata={"source":"localhost:4657468"}
)
print(sample_doc)
type(sample_doc)

page_content='Hello World !' metadata={'source': 'localhost:4657468'}


langchain_core.documents.base.Document

### Text Data

In [5]:
from langchain_community.document_loaders.text import TextLoader

loader = TextLoader("data/Python.txt", encoding="utf-8")

document = loader.load()

In [6]:
print(document[0].metadata)

{'source': 'data/Python.txt'}


### PDF Data

In [29]:
from langchain_community.document_loaders.pdf import PyPDFLoader, PyMuPDFLoader

loader = PyPDFLoader(file_path="data/research.pdf")
document = loader.load()
document[0].metadata

{'producer': 'pdfcpu v0.12.1 dev',
 'creator': 'PyPDF',
 'creationdate': '2026-07-27T10:16:33+00:00',
 'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin',
 'book': 'Advances in Neural Information Processing Systems 30',
 'created': '2017',
 'date': '2017',
 'description': 'Paper accepted and presented at the Neural Information Processing Systems Conference (http://nips.cc/)',
 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and 

In [30]:
document[0].page_content

'Attention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszkaiser@google.com\nIllia Polosukhin∗‡\nillia.polosukhin@gmail.com\nAbstract\nThe dominant sequence transduction models are based on complex recurrent or\nconvolutional neural networks that include an encoder and a decoder. The best\nperforming models also connect the encoder and decoder through an attention\nmechanism. We propose a new simple network architecture, the Transformer,\nbased solely on attention mechanisms, dispensing with recurrence and convolutions\nentirely. Experiments on two machine translation tasks show these models to\nbe superior in quality while being more parallelizable and requiring signiﬁcantly

# 2. Ingestion Pipeline
- Data -> Documents

In [31]:
import os
from langchain_community.document_loaders.pdf import PyPDFLoader

In [32]:
def load_all_pdfs():
    folder_path = "data/pdfs"
    num_docs = 0
    all_docs = []

    for file_name in os.listdir(folder_path):
        if file_name.lower().endswith(".pdf"):
            pdf_path = os.path.join(folder_path, file_name)

            loader = PyPDFLoader(pdf_path)
            doc = loader.load()

            all_docs.extend(doc)
            num_docs += 1

    print("total PDFs:", num_docs)
    print("total Pages:", len(all_docs))
    return all_docs

In [33]:
all_pdf_documents = load_all_pdfs()

total PDFs: 2
total Pages: 32


## 3. Chunks
- Data -> Documents -> Chunks

In [34]:
# !pip install langchain_text_splitters

In [35]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_doc(documents, chunk_size=500, chunk_overlap=50):
    text_splitters = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )
    chunked_docs = text_splitters.split_documents(documents)
    return chunked_docs

In [36]:
chunks = split_doc(all_pdf_documents)

In [37]:
len(chunks)

321

In [38]:
print(chunks[0].page_content)

Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or


## 4. Embeddings
- Data -> Documents -> Chunks -> Embeddings

In [39]:
from sentence_transformers import SentenceTransformer

In [40]:
class EmbeddingManager():
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model_name = model_name
        print("Loading Model...", self.model_name)
        self.model = SentenceTransformer(self.model_name)
        print("Embedding Dimensions...", self.model.get_embedding_dimension())

    def generate_embeddings(self, text):
        embeddings = self.model.encode(text, show_progress_bar=True)

        print("Embedding shape", embeddings.shape)
        return embeddings


In [41]:
embedding_manager = EmbeddingManager()

Loading Model... all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\mdmea\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\mdmea\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding Dimensions... 384


## 5. Store Embeddings in Vector DB
- Data -> Documents -> Chunks -> Embeddings -> Vector DB

In [42]:
import chromadb
import uuid

In [43]:
class VectorStoreManager():
    def __init__(self, persist_directory="data/vector_store", collection_name="pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None

        self._initilize_store()

    def _initilize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)
        # create a client
        self.client = chromadb.PersistentClient(path=self.persist_directory)
        #create the collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description":"Vector store collection for pdf Embeddings in RAG"}
            )
        print("Initialize the vector storage with collection:", self.collection_name)
        print("docs in collection:", self.collection.count())

    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("num of documents does not match num of embeddings")
        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            #creates ids
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            # meta data
            metadata = dict(doc.metadata)
            metadata['doc_idx'] = i
            metadata['content_length'] = len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)
            embeddings_list.append(embedding.tolist())

            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=all_metadata,
                documents=documents_content
            )
        print("Total doc added in vector store:", len(documents_content))
        print("docs in collection:", self.collection.count())

In [44]:
vector_store = VectorStoreManager()

Initialize the vector storage with collection: pdf_documents
docs in collection: 0


In [45]:
texts = [doc.page_content for doc in chunks]
embeddings = embedding_manager.generate_embeddings(texts)
print(embeddings)
vector_store.add_documents(chunks, embeddings)

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

Embedding shape (321, 384)
[[-0.06644694 -0.1013386   0.06421992 ...  0.08527189  0.01939096
  -0.03197717]
 [-0.04218758 -0.10582767  0.00103702 ...  0.05450409 -0.0449203
  -0.02199423]
 [-0.02385589 -0.10134074  0.02038444 ... -0.00769125 -0.10234489
  -0.00285462]
 ...
 [-0.04638284 -0.07808162 -0.03579675 ...  0.04963214 -0.08682774
  -0.06420165]
 [ 0.02013829 -0.04678571  0.02046878 ...  0.0790542   0.00145204
   0.03949955]
 [-0.06597345  0.00491244 -0.02753688 ... -0.02634121 -0.00415465
  -0.00835984]]
Total doc added in vector store: 321
docs in collection: 321


## 6. Retrieval Pipeline

In [46]:
from sklearn.metrics.pairwise import cosine_similarity

In [47]:
class RAGRetriever():
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store

    def retrieve(self, query, top_k=5, score_threshold=0.0):
        # query -> embeddings
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        # semantic search
        results = self.vector_store.collection.query(
            query_embeddings = [query_embeddings.tolist()],
            n_results = top_k
        )
        #cosine similarity
        retrieve_docs = []
        if results['documents'] and results['documents'][0]:
            ids = results['ids'][0]
            metadatas = results['metadatas'][0]
            documents = results['documents'][0]
            distances = results['distances'][0]

            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                similarity_score = 1- distance
                if similarity_score >= score_threshold:
                    retrieve_docs.append({
                        "id":doc_id,
                        "document":document,
                        "metadata":metadata,
                        "distance":distance,
                        "similarity_socre": similarity_score,
                        "rank": i + 1
                    })
            print(f"retrieved {len(retrieve_docs)} documents")

        else:
            print("no document found")

        return retrieve_docs

In [48]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [49]:
rag_retriever.retrieve("what is encoder RAG?")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding shape (1, 384)
retrieved 5 documents


[{'id': 'doc_255f763b-8694-4123-a0bc-3b50ed3ca65a',
  'document': 'and speculate on upcoming trends and innovations.\nOur contributions are as follows:\n• In this survey, we present a thorough and systematic\nreview of the state-of-the-art RAG methods, delineating\nits evolution through paradigms including naive RAG,\narXiv:2312.10997v5  [cs.CL]  27 Mar 2024',
  'metadata': {'page_label': '1',
   'subject': '',
   'source': 'data/pdfs\\research2.pdf',
   'total_pages': 21,
   'trapped': '/False',
   'producer': 'pdfTeX-1.40.25',
   'creator': 'LaTeX with hyperref',
   'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
   'moddate': '2024-03-28T00:54:45+00:00',
   'author': '',
   'content_length': 288,
   'keywords': '',
   'title': '',
   'creationdate': '2024-03-28T00:54:45+00:00',
   'page': 0,
   'doc_idx': 88},
  'distance': 0.763151228427887,
  'similarity_socre': 0.23684877157211304,
  'rank': 1},
 {'id': 'doc_4e94b26f-61

## 7. Intregate with LLM

OpenAI - GPT
- !pip install langchain-openai
- !pip install langchain-groq

In [ ]:
from dotenv import load_dotenv
# Load environment variables from the .env file
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [69]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
    openai_api_key=OPENAI_API_KEY,
    model="gpt-5.6",
    temperature=0.1, # creativity value
    max_tokens=1024
)

In [72]:
def generate_output(query, rag_retriever, llm , top_k=5):
    results = rag_retriever.retrieve(query, top_k)
    context = "\n".join(doc["document"] for doc in results) if results else ""

    if not context:
        print("we don't found relevant context for the given query.")

    prompt = f"""use given context to generate answer for the query Context:{context} Query:{query}"""
    response = llm.invoke(prompt) # expecting string as prompt
    return response.content



In [73]:
answer = generate_output("what is RAG?", rag_retriever, llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding shape (1, 384)
retrieved 5 documents


RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

Groq API

In [101]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    groq_api_key=GROQ_API_KEY,
    model="qwen/qwen3.6-27b",
    temperature=0.1, # creativity value
    max_tokens=4096,
    reasoning_format="parsed"
)

In [102]:
def generate_output(query, rag_retriever, llm , top_k=5):
    results = rag_retriever.retrieve(query, top_k)
    context = "\n".join(doc["document"] for doc in results) if results else ""

    if not context:
        print("we don't found relevant context for the given query.")

    prompt = f"""use given context to generate answer for the query\nContext: {context}\nQuery: {query}"""
    response = llm.invoke([prompt]) # expecting list as prompt
    return response.content

In [105]:
answer = generate_output("what is encoder decoder?", rag_retriever, llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding shape (1, 384)
retrieved 5 documents


In [106]:
print(answer)

Based on the provided context, the **Encoder** and **Decoder** are the two core stacked components of the Transformer architecture:

* **Encoder:** Composed of a stack of `N = 6` identical layers. Each layer contains two sub-layers: the first is a multi-head self-attention mechanism, and the second is a simple position-wise component. It employs residual connections and layer normalization. In the encoder, self-attention allows every position to attend to all positions in the previous layer.
* **Decoder:** Also composed of a stack of `N = 6` identical layers, but each layer contains three sub-layers. It includes the same two sub-layers as the encoder, plus a third sub-layer that performs multi-head attention over the encoder's output. It also uses residual connections and layer normalization. The decoder's self-attention is modified (masked) to prevent each position from attending to future positions, enabling auto-regressive sequence generation.

Both components share a model dimensio